# Grok-nlp-neural-from-scratch

**S06–S10** on Kaggle **T4×2**: neural text classification → BiLSTM NER → RNN LM → Seq2Seq MT → Tiny Transformer.

Each stage: concept → train → real I/O → compare previous → save JSON under `/kaggle/working`.


In [ ]:
import os, json, math, random, re, time, platform
from pathlib import Path
from collections import Counter

os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
RESULTS = {}

print('python', platform.python_version())
print('torch', torch.__version__)
print('device', DEVICE, 'n_gpu', N_GPU)
for i in range(N_GPU):
    print(f'  gpu{i}', torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'Need GPU (T4×2)'



In [ ]:
# Shared tiny datasets
CLS_TRAIN = [
    ("I love this movie amazing acting", "pos"),
    ("Fantastic film highly recommend", "pos"),
    ("Wonderful cinema masterpiece", "pos"),
    ("Great story superb performances", "pos"),
    ("Enjoyed every minute brilliant", "pos"),
    ("This movie is terrible boring", "neg"),
    ("Awful film waste of time", "neg"),
    ("Horrible acting ridiculous plot", "neg"),
    ("I hate this complete disaster", "neg"),
    ("Boring poorly written script", "neg"),
    ("Stock market rose after earnings", "business"),
    ("Investors buy shares strong growth", "business"),
    ("Company profits beat estimates", "business"),
    ("Bank raises interest rates today", "business"),
    ("Goal scored seals victory", "sports"),
    ("Team won championship final", "sports"),
    ("Athlete breaks world record", "sports"),
    ("Coach praises defense clean sheet", "sports"),
]
CLS_TEST = [
    ("I really love amazing plot", "pos"),
    ("Terrible boring waste money", "neg"),
    ("Shares climb after profits", "business"),
    ("Team wins final championship", "sports"),
]
NER_DATA = [
    (["John","lives","in","Paris"], ["B-PER","O","O","B-LOC"]),
    (["Mary","works","at","Google"], ["B-PER","O","O","B-ORG"]),
    (["Paris","is","in","France"], ["B-LOC","O","O","B-LOC"]),
    (["Apple","hired","Tim","Cook"], ["B-ORG","O","B-PER","I-PER"]),
    (["Berlin","hosts","many","startups"], ["B-LOC","O","O","O"]),
    (["Alice","visited","London","yesterday"], ["B-PER","O","B-LOC","O"]),
    (["Microsoft","is","based","in","Seattle"], ["B-ORG","O","O","O","B-LOC"]),
    (["Bob","met","Carol","in","Tokyo"], ["B-PER","O","B-PER","O","B-LOC"]),
    (["John","visited","Berlin"], ["B-PER","O","B-LOC"]),
    (["Google","hired","Alice"], ["B-ORG","O","B-PER"]),
]
MT_PAIRS = [
    ("hello","bonjour"), ("goodbye","au revoir"), ("thank you","merci"),
    ("good morning","bonjour"), ("I love cats","j aime les chats"),
    ("I love dogs","j aime les chiens"), ("the cat sleeps","le chat dort"),
    ("the dog runs","le chien court"), ("she reads a book","elle lit un livre"),
    ("he writes a letter","il ecrit une lettre"), ("where is the station","ou est la gare"),
    ("how are you","comment allez vous"), ("I am fine","je vais bien"),
    ("see you tomorrow","a demain"), ("the weather is nice","il fait beau"),
    ("please help me","aidez moi"), ("I like music","j aime la musique"),
    ("the book is red","le livre est rouge"), ("we eat bread","nous mangeons du pain"),
    ("they play football","ils jouent au football"),
]

def tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def save_stage(name, payload):
    RESULTS[name] = payload
    p = OUT / f'{name}.json'
    p.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    print('saved', p)



## S06 · Neural Text Classification (Embedding + CNN)

**Concept**: learn word vectors + local n-gram filters end-to-end.  
**vs S01**: no handcrafted TF-IDF; gradients shape features for the label.


In [ ]:
# ===== S06 Neural CLS =====
print('='*60, '\nS06 Neural Text Classification')
words = sorted({w for t,_ in CLS_TRAIN for w in tok(t)})
stoi = {w:i+1 for i,w in enumerate(words)}  # 0=pad
itos = {i:w for w,i in stoi.items()}
labels = sorted({y for _,y in CLS_TRAIN})
l2i = {l:i for i,l in enumerate(labels)}
MAXLEN = 12

def encode(text):
    ids = [stoi.get(w, 0) for w in tok(text)][:MAXLEN]
    ids += [0]*(MAXLEN-len(ids))
    return ids

X = torch.tensor([encode(t) for t,_ in CLS_TRAIN], dtype=torch.long)
y = torch.tensor([l2i[l] for _,l in CLS_TRAIN], dtype=torch.long)
Xt = torch.tensor([encode(t) for t,_ in CLS_TEST], dtype=torch.long)
yt = torch.tensor([l2i[l] for _,l in CLS_TEST], dtype=torch.long)

class TextCNN(nn.Module):
    def __init__(self, V, C, emb=64, filters=64):
        super().__init__()
        self.emb = nn.Embedding(V+1, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, filters, k) for k in (2,3,4)])
        self.fc = nn.Linear(filters*3, C)
    def forward(self, x):
        # x: B,T
        e = self.emb(x).transpose(1,2)  # B,E,T
        xs = [torch.relu(conv(e)).max(dim=-1).values for conv in self.convs]
        return self.fc(torch.cat(xs, dim=-1))

model = TextCNN(len(stoi), len(labels)).to(DEVICE)
if N_GPU >= 2:
    model = nn.DataParallel(model)
    print('DataParallel on', N_GPU, 'GPUs')
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
loader = DataLoader(TensorDataset(X,y), batch_size=8, shuffle=True)

hist=[]
t0=time.time()
for epoch in range(1, 81):
    model.train(); total=0; correct=0; loss_sum=0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward(); opt.step()
        loss_sum += loss.item()*len(xb)
        correct += (logits.argmax(1)==yb).sum().item(); total += len(xb)
    hist.append({'epoch':epoch,'loss':loss_sum/total,'acc':correct/total})
    if epoch % 20 == 0:
        print(f'epoch {epoch} loss={hist[-1]["loss"]:.3f} acc={hist[-1]["acc"]:.3f}')
train_s = time.time()-t0

model.eval()
with torch.no_grad():
    pred = model(Xt.to(DEVICE)).argmax(1).cpu()
test_acc = (pred==yt).float().mean().item()
print('test_acc', test_acc)

demos=[]
with torch.no_grad():
    for text,gold in CLS_TEST:
        logits = model(torch.tensor([encode(text)], device=DEVICE))
        p = F.softmax(logits, dim=-1)[0]
        ip = int(p.argmax())
        demos.append({'input':text,'gold':gold,'pred':labels[ip],
                      'probs':{labels[i]:float(p[i]) for i in range(len(labels))}})
        print(f'  {text!r} → {labels[ip]} (gold={gold})')

save_stage('S06_neural_cls', {
    'concept':'Embedding+TextCNN end-to-end classification',
    'metric':{'test_acc':test_acc,'train_time_s':train_s,'final_train_acc':hist[-1]['acc']},
    'vs_previous':'S01 handcrafted TF-IDF; S06 learns features from labels',
    'demos':demos,'history_tail':hist[-5:],
    'new_capability':'Neural text classification with learned embeddings',
    'device_count':N_GPU,
})
# keep vocab for later
S06 = {'stoi':stoi,'labels':labels,'encode':encode,'model':model}



## S07 · BiLSTM Token Classification (NER)

**Concept**: bidirectional LSTM emits BIO tags per token.  
**vs S06**: prediction is a **sequence of labels**, not one doc label.


In [ ]:
# ===== S07 BiLSTM NER =====
print('='*60, '\nS07 BiLSTM Token Classification (NER)')
ner_words = sorted({w for toks,_ in NER_DATA for w in toks})
ner_tags = sorted({t for _,tags in NER_DATA for t in tags})
nw2i = {w:i+1 for i,w in enumerate(ner_words)}; nw2i['<unk>']=len(nw2i)+1
nt2i = {t:i for i,t in enumerate(ner_tags)}; i2nt={i:t for t,i in nt2i.items()}
MAXT=6

def enc_ner(toks):
    ids=[nw2i.get(w, nw2i['<unk>']) for w in toks][:MAXT]
    ids += [0]*(MAXT-len(ids)); return ids
def enc_tags(tags):
    ids=[nt2i[t] for t in tags][:MAXT]
    # pad with O if exists else 0
    o_id = nt2i.get('O', 0)
    ids += [o_id]*(MAXT-len(ids)); return ids

Xn=torch.tensor([enc_ner(t) for t,_ in NER_DATA], dtype=torch.long)
Yn=torch.tensor([enc_tags(y) for _,y in NER_DATA], dtype=torch.long)
# mask real lengths
lens=[len(t) for t,_ in NER_DATA]

class BiLSTMNer(nn.Module):
    def __init__(self, V, T, emb=64, hid=64):
        super().__init__()
        self.emb=nn.Embedding(V+2, emb, padding_idx=0)
        self.lstm=nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.fc=nn.Linear(hid*2, T)
    def forward(self,x):
        e=self.emb(x)
        h,_=self.lstm(e)
        return self.fc(h)

ner = BiLSTMNer(len(nw2i), len(nt2i)).to(DEVICE)
if N_GPU>=2:
    ner = nn.DataParallel(ner)
opt=torch.optim.Adam(ner.parameters(), lr=2e-3)
crit=nn.CrossEntropyLoss()
for epoch in range(1, 121):
    ner.train(); opt.zero_grad()
    logits=ner(Xn.to(DEVICE))  # B,T,C
    loss=crit(logits.reshape(-1, len(nt2i)), Yn.to(DEVICE).reshape(-1))
    loss.backward(); opt.step()
    if epoch%30==0:
        pred=logits.argmax(-1).cpu()
        # token acc on non-pad positions roughly
        acc=(pred==Yn).float().mean().item()
        print(f'epoch {epoch} loss={loss.item():.3f} token_acc~{acc:.3f}')

ner.eval()
demos=[]
with torch.no_grad():
    for toks, gold in NER_DATA[-4:]:
        x=torch.tensor([enc_ner(toks)], device=DEVICE)
        pred=ner(x).argmax(-1)[0].cpu().tolist()[:len(toks)]
        tags=[i2nt[i] for i in pred]
        demos.append({'tokens':toks,'gold':gold,'pred':tags})
        print(' ', list(zip(toks, tags)), 'gold', gold)

# entity-level exact match on held-out style last 2
ok=0
for d in demos[-2:]:
    ok += int(d['pred']==d['gold'])
ent_acc = ok/2
print('heldout exact-seq', ent_acc)

save_stage('S07_bilstm_ner', {
    'concept':'BiLSTM sequence labeling with BIO tags',
    'metric':{'heldout_exact_seq':ent_acc,'n_tags':len(nt2i)},
    'vs_previous':'S06 one label/doc; S07 label per token (NER)',
    'demos':demos,
    'new_capability':'Token classification / named entity recognition',
    'tags':ner_tags,
})



## S08 · Character RNN Language Model → Text Generation

**Concept**: next-char prediction; sample to generate.  
**vs classification**: unsupervised density model of text.


In [ ]:
# ===== S08 Char RNN LM =====
print('='*60, '\nS08 Char-level RNN Language Model / Generation')
corpus = ' '.join(t for t,_ in CLS_TRAIN) + ' ' + ' '.join(a+' '+b for a,b in MT_PAIRS)
corpus = corpus.lower()
chars = sorted(set(corpus))
c2i = {c:i for i,c in enumerate(chars)}
i2c = {i:c for c,i in c2i.items()}
print('chars', len(chars), 'corpus_len', len(corpus))

SEQ=40
xs, ys = [], []
for i in range(0, len(corpus)-SEQ-1, 3):
    xs.append([c2i[c] for c in corpus[i:i+SEQ]])
    ys.append([c2i[c] for c in corpus[i+1:i+SEQ+1]])
Xlm = torch.tensor(xs, dtype=torch.long)
Ylm = torch.tensor(ys, dtype=torch.long)

class CharRNN(nn.Module):
    def __init__(self, V, emb=64, hid=128):
        super().__init__()
        self.emb=nn.Embedding(V, emb)
        self.rnn=nn.LSTM(emb, hid, batch_first=True, num_layers=2)
        self.fc=nn.Linear(hid, V)
    def forward(self, x, hc=None):
        e=self.emb(x)
        out, hc = self.rnn(e, hc)
        return self.fc(out), hc

lm = CharRNN(len(chars)).to(DEVICE)
if N_GPU>=2:
    lm = nn.DataParallel(lm)
opt=torch.optim.Adam(lm.parameters(), lr=2e-3)
loader=DataLoader(TensorDataset(Xlm,Ylm), batch_size=32, shuffle=True)
for epoch in range(1, 31):
    lm.train(); tot=0; n=0
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        # DataParallel wraps
        logits, _ = (lm.module if isinstance(lm, nn.DataParallel) else lm)(xb)
        loss=F.cross_entropy(logits.reshape(-1,len(chars)), yb.reshape(-1))
        loss.backward(); opt.step()
        tot+=loss.item()*len(xb); n+=len(xb)
    if epoch%10==0:
        print(f'epoch {epoch} loss={tot/n:.3f} ppl={math.exp(min(tot/n,20)):.2f}')

def generate(prefix, n=80, temperature=0.8):
    model_ = lm.module if isinstance(lm, nn.DataParallel) else lm
    model_.eval()
    s = prefix.lower()
    ids = [c2i.get(c, 0) for c in s[-SEQ:]]
    hc=None
    with torch.no_grad():
        for _ in range(n):
            x=torch.tensor([ids[-SEQ:]], device=DEVICE)
            logits, hc = model_(x, hc)
            logits = logits[0,-1]/temperature
            probs=F.softmax(logits, dim=-1)
            nxt=torch.multinomial(probs, 1).item()
            ids.append(nxt)
            s += i2c[nxt]
    return s

gens=[]
for pref in ['i love ', 'the team ', 'bonjour ', 'stock ']:
    g=generate(pref, n=60)
    gens.append({'prefix':pref, 'generated':g})
    print('GEN', g)

save_stage('S08_rnn_lm', {
    'concept':'Char LSTM language model + sampling',
    'metric':{'final_loss':tot/n, 'vocab_chars':len(chars)},
    'vs_previous':'S06/S07 discriminative; S08 generative density model',
    'demos':gens,
    'new_capability':'Open-ended text generation from a learned LM',
})



## S09 · Seq2Seq + Attention (Translation en→fr)

**Concept**: encoder reads source; decoder attends and emits target tokens.  
**vs S08**: conditional generation (MT).


In [ ]:
# ===== S09 Seq2Seq Attention MT =====
print('='*60, '\nS09 Seq2Seq + Attention Translation')
# Build joint word vocab
src_sents=[tok(a) for a,_ in MT_PAIRS]
tgt_sents=[tok(b) for _,b in MT_PAIRS]
PAD,BOS,EOS,UNK=0,1,2,3
vocab=sorted({w for s in src_sents+tgt_sents for w in s})
w2i={'<pad>':0,'<bos>':1,'<eos>':2,'<unk>':3}
for w in vocab: w2i[w]=len(w2i)
i2w={i:w for w,i in w2i.items()}
VS=len(w2i)
MAX_S, MAX_T = 8, 8

def enc_src(words):
    ids=[w2i.get(w,UNK) for w in words][:MAX_S]
    ids += [PAD]*(MAX_S-len(ids)); return ids
def enc_tgt(words):
    ids=[BOS]+[w2i.get(w,UNK) for w in words][:MAX_T-2]+[EOS]
    ids += [PAD]*(MAX_T-len(ids)); return ids[:MAX_T]

Xs=torch.tensor([enc_src(s) for s in src_sents], dtype=torch.long)
Yt=torch.tensor([enc_tgt(s) for s in tgt_sents], dtype=torch.long)

class Encoder(nn.Module):
    def __init__(self, V, emb=64, hid=128):
        super().__init__()
        self.emb=nn.Embedding(V, emb, padding_idx=0)
        self.rnn=nn.GRU(emb, hid, batch_first=True, bidirectional=True)
        self.proj=nn.Linear(hid*2, hid)
    def forward(self, x):
        e=self.emb(x)
        out,_=self.rnn(e)
        return self.proj(out)  # B,T,H

class Attention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        self.W=nn.Linear(hid, hid)
    def forward(self, dec_h, enc_out, mask=None):
        # dec_h: B,H  enc_out:B,S,H
        scores = torch.bmm(self.W(enc_out), dec_h.unsqueeze(-1)).squeeze(-1)  # B,S
        if mask is not None:
            scores = scores.masked_fill(mask==0, -1e9)
        w=F.softmax(scores, dim=-1)
        ctx=torch.bmm(w.unsqueeze(1), enc_out).squeeze(1)
        return ctx, w

class Decoder(nn.Module):
    def __init__(self, V, emb=64, hid=128):
        super().__init__()
        self.emb=nn.Embedding(V, emb, padding_idx=0)
        self.rnn=nn.GRU(emb+hid, hid, batch_first=True)
        self.attn=Attention(hid)
        self.fc=nn.Linear(hid*2, V)
    def step(self, y_prev, h, enc_out, mask):
        e=self.emb(y_prev)  # B,E
        ctx, w = self.attn(h.squeeze(0), enc_out, mask)
        inp=torch.cat([e, ctx], dim=-1).unsqueeze(1)
        out,h = self.rnn(inp, h)
        logits=self.fc(torch.cat([out.squeeze(1), ctx], dim=-1))
        return logits, h, w

class Seq2Seq(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.enc=Encoder(V)
        self.dec=Decoder(V)
        self.hid=128
    def forward(self, src, tgt):
        enc_out=self.enc(src)
        mask=(src!=0).float()
        B=src.size(0)
        h=torch.zeros(1,B,self.hid, device=src.device)
        logits_all=[]
        y=tgt[:,0]
        for t in range(1, tgt.size(1)):
            logits,h,_=self.dec.step(y, h, enc_out, mask)
            logits_all.append(logits)
            # teacher forcing
            y=tgt[:,t]
        return torch.stack(logits_all, dim=1)

s2s=Seq2Seq(VS).to(DEVICE)
opt=torch.optim.Adam(s2s.parameters(), lr=3e-3)
for epoch in range(1, 201):
    s2s.train(); opt.zero_grad()
    src,tgt=Xs.to(DEVICE), Yt.to(DEVICE)
    logits=s2s(src,tgt)  # B,T-1,V
    loss=F.cross_entropy(logits.reshape(-1,VS), tgt[:,1:].reshape(-1), ignore_index=PAD)
    loss.backward(); opt.step()
    if epoch%50==0:
        print(f'epoch {epoch} loss={loss.item():.3f}')

@torch.no_grad()
def translate(sentence, max_len=8):
    s2s.eval()
    src=torch.tensor([enc_src(tok(sentence))], device=DEVICE)
    enc_out=s2s.enc(src)
    mask=(src!=0).float()
    h=torch.zeros(1,1,s2s.hid, device=DEVICE)
    y=torch.tensor([BOS], device=DEVICE)
    out_ids=[]
    for _ in range(max_len):
        logits,h,_=s2s.dec.step(y, h, enc_out, mask)
        y=logits.argmax(-1)
        if y.item()==EOS: break
        if y.item() not in (PAD,BOS):
            out_ids.append(y.item())
        y=y
    return ' '.join(i2w[i] for i in out_ids)

demos=[]
for src,gold in MT_PAIRS[:8]+[("I love cats","j aime les chats"),("the dog runs","le chien court")]:
    hyp=translate(src)
    demos.append({'src':src,'gold':gold,'hyp':hyp})
    print(f'  {src!r} → {hyp!r}  (gold={gold!r})')

# rough token accuracy on training pairs
tok_hits=0; tok_tot=0
for src,gold in MT_PAIRS:
    hyp=set(tok(translate(src))); g=set(tok(gold))
    tok_hits += len(hyp & g); tok_tot += max(1,len(g))
overlap=tok_hits/tok_tot
print('mean gold-token recall', overlap)

save_stage('S09_seq2seq_mt', {
    'concept':'Bidirectional GRU encoder + GRU decoder with attention',
    'metric':{'final_loss':float(loss.item()), 'token_recall':overlap},
    'vs_previous':'S08 unconditional LM; S09 conditional MT generation',
    'demos':demos,
    'new_capability':'Neural machine translation (en→fr mini)',
})



## S10 · Transformer from Scratch (fill-mask + features + generation)

**Concept**: multi-head self-attention + FFN.  
MLM training enables **fill-mask**; pooled states = **features**; causal decode = **generation**.


In [ ]:
# ===== S10 Tiny Transformer =====
print('='*60, '\nS10 Tiny Transformer (MLM + causal gen)')
# word-level on CLS+MT corpus
all_sents = [tok(t) for t,_ in CLS_TRAIN] + [tok(a) for a,_ in MT_PAIRS] + [tok(b) for _,b in MT_PAIRS]
vocab=sorted({w for s in all_sents for w in s})
w2i={'<pad>':0,'<mask>':1,'<bos>':2,'<eos>':3,'<unk>':4}
for w in vocab: w2i[w]=len(w2i)
i2w={i:w for w,i in w2i.items()}
V=len(w2i); D=64; L=16; H=4; DEPTH=2

def enc_words(words, max_len=L):
    ids=[w2i.get(w, w2i['<unk>']) for w in words][:max_len]
    ids += [0]*(max_len-len(ids)); return ids

class MultiHead(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d%h==0
        self.h=h; self.dk=d//h
        self.qkv=nn.Linear(d, 3*d)
        self.proj=nn.Linear(d,d)
    def forward(self, x, mask=None, causal=False):
        B,T,D=x.shape
        qkv=self.qkv(x).reshape(B,T,3,self.h,self.dk).permute(2,0,3,1,4)
        q,k,v=qkv[0],qkv[1],qkv[2]
        att = (q @ k.transpose(-2,-1)) / math.sqrt(self.dk)
        if causal:
            causal_mask=torch.triu(torch.ones(T,T, device=x.device), diagonal=1).bool()
            att=att.masked_fill(causal_mask, -1e9)
        if mask is not None:
            att=att.masked_fill(mask[:,None,None,:]==0, -1e9)
        w=F.softmax(att, dim=-1)
        out=(w@v).transpose(1,2).reshape(B,T,D)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1=nn.LayerNorm(d); self.ln2=nn.LayerNorm(d)
        self.att=MultiHead(d,h)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Linear(4*d,d))
    def forward(self, x, mask=None, causal=False):
        x=x+self.att(self.ln1(x), mask=mask, causal=causal)
        x=x+self.ff(self.ln2(x)); return x

class TinyTransformer(nn.Module):
    def __init__(self, V, d=D, n=DEPTH, h=H, max_len=L):
        super().__init__()
        self.emb=nn.Embedding(V, d, padding_idx=0)
        self.pos=nn.Embedding(max_len, d)
        self.blocks=nn.ModuleList([Block(d,h) for _ in range(n)])
        self.ln=nn.LayerNorm(d)
        self.head=nn.Linear(d, V)
    def forward(self, x, causal=False):
        B,T=x.shape
        pos=torch.arange(T, device=x.device)
        h=self.emb(x)+self.pos(pos)[None,:]
        mask=(x!=0).float()
        for b in self.blocks:
            h=b(h, mask=mask, causal=causal)
        h=self.ln(h)
        return self.head(h), h

tfm=TinyTransformer(V).to(DEVICE)
opt=torch.optim.Adam(tfm.parameters(), lr=3e-3)

def make_mlm_batch(batch_sents):
    xs=[]; ys=[]
    for words in batch_sents:
        ids=enc_words(words)
        y=ids.copy()
        # mask 15% non-pad
        for i,tid in enumerate(ids):
            if tid==0: continue
            if random.random()<0.15:
                ids[i]=w2i['<mask>']
        xs.append(ids); ys.append(y)
    return torch.tensor(xs), torch.tensor(ys)

for epoch in range(1, 121):
    tfm.train()
    random.shuffle(all_sents)
    tot=0; n=0
    for i in range(0, len(all_sents), 8):
        batch=all_sents[i:i+8]
        if not batch: continue
        x,y=make_mlm_batch(batch)
        x,y=x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits, _ = tfm(x, causal=False)
        # only masked positions
        mask=(x==w2i['<mask>'])
        if mask.sum()==0: continue
        loss=F.cross_entropy(logits[mask], y[mask])
        loss.backward(); opt.step()
        tot+=loss.item(); n+=1
    if epoch%30==0:
        print(f'MLM epoch {epoch} loss={tot/max(n,1):.3f}')

# Fill-mask demos
@torch.no_grad()
def fill_mask(words, mask_index):
    tfm.eval()
    ids=enc_words(words)
    ids[mask_index]=w2i['<mask>']
    x=torch.tensor([ids], device=DEVICE)
    logits,_=tfm(x)
    pred=logits[0, mask_index].argmax().item()
    return i2w.get(pred, '<unk>')

fill_demos=[]
for words, mi in [(['i','love','this','movie'], 1), (['team','won','championship','final'], 1), (['the','cat','sleeps'], 1)]:
    pred=fill_mask(words, mi)
    filled=words.copy(); filled[mi]=pred
    fill_demos.append({'template':words, 'mask_index':mi, 'pred':pred, 'filled':filled})
    print('FILL', words, '→', filled)

# Feature extraction: cosine between sentence means
@torch.no_grad()
def sent_feat(text):
    tfm.eval()
    x=torch.tensor([enc_words(tok(text))], device=DEVICE)
    _, h = tfm(x)
    mask=(x!=0).float().unsqueeze(-1)
    v=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
    v=F.normalize(v, dim=-1)
    return v.squeeze(0)

pairs=[('I love this amazing movie','Fantastic film recommend'),
       ('I love this amazing movie','Team won championship final'),
       ('photosynthesis plants sunlight','plants convert sunlight energy')]
# photosynthesis may be OOV — use in-vocab
pairs=[('i love this movie','fantastic film wonderful'),
       ('i love this movie','team won championship'),
       ('investors buy shares','company profits growth')]
feat_demos=[]
for a,b in pairs:
    fa,fb=sent_feat(a), sent_feat(b)
    c=float((fa@fb).item())
    feat_demos.append({'a':a,'b':b,'cos':c})
    print(f'FEAT cos={c:.3f} | {a} || {b}')

# Causal generation (fine-tune briefly)
for epoch in range(1, 41):
    tfm.train(); tot=0; n=0
    for words in all_sents:
        ids=[w2i['<bos>']]+ [w2i.get(w,4) for w in words][:L-2]+[w2i['<eos>']]
        ids += [0]*(L-len(ids)); ids=ids[:L]
        x=torch.tensor([ids[:-1]], device=DEVICE)
        y=torch.tensor([ids[1:]], device=DEVICE)
        opt.zero_grad()
        logits,_=tfm(x, causal=True)
        loss=F.cross_entropy(logits.reshape(-1,V), y.reshape(-1), ignore_index=0)
        loss.backward(); opt.step()
        tot+=loss.item(); n+=1
    if epoch%20==0: print(f'CAUSAL epoch {epoch} loss={tot/n:.3f}')

@torch.no_grad()
def gen_tfm(prefix_words, n=8):
    tfm.eval()
    ids=[w2i['<bos>']]+[w2i.get(w,4) for w in prefix_words]
    for _ in range(n):
        x=torch.tensor([ids[-L:]+[0]*(L-len(ids[-L:]))], device=DEVICE)
        # only use actual length positions — simpler: feed pad-trimmed
        x=torch.tensor([ (ids + [0]*L)[:L] ], device=DEVICE)
        logits,_=tfm(x, causal=True)
        pos=min(len(ids)-1, L-1)
        nxt=logits[0,pos].argmax().item()
        if nxt in (0, w2i['<eos>']): break
        ids.append(nxt)
    return [i2w.get(i,'?') for i in ids[1:]]

gen_demos=[]
for pref in [['i','love'], ['the','team'], ['stock','market']]:
    g=gen_tfm(pref)
    gen_demos.append({'prefix':pref,'generated':g})
    print('GEN', g)

save_stage('S10_tiny_transformer', {
    'concept':'Multi-head Transformer encoder trained with MLM + causal LM',
    'metric':{'V':V,'d':D,'layers':DEPTH,'heads':H},
    'vs_previous':'S09 RNN attention; S10 pure self-attention, bidirectional MLM',
    'fill_mask':fill_demos,
    'features':feat_demos,
    'generation':gen_demos,
    'new_capability':'Fill-mask, feature extraction, and generation in one Transformer',
})



In [ ]:
# Aggregate neural stages
summary = {
    'stages': list(RESULTS.keys()),
    'gpu': {'count': N_GPU, 'names': [torch.cuda.get_device_name(i) for i in range(N_GPU)]},
    'torch': torch.__version__,
    'results': RESULTS,
}
(OUT/'nlp_neural_from_scratch_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print('ALL NEURAL STAGES DONE', list(RESULTS.keys()))
print('devices', summary['gpu'])

